# Phase 5A — frozen official BIFOLD S1+S2 validation control

This notebook performs validation-only inference with the official 12-channel checkpoint. It never trains, adapts, or opens sealed test pixels.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/SIH-26167-SATQuery')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', os.environ['SATQUERY_REPO_URL'], str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[multisensor]', 'appdirs>=1.4.4,<2', 'lightning>=2,<3', 'lmdb==1.6.2', 'timm==0.9.16'], check=True)
# ConfigILM 0.7.0 is pure Python, but its legacy metadata excludes Kaggle Python 3.12.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--ignore-requires-python', 'configilm==0.7.0'], check=True)
REBEN_DIR = Path('/kaggle/working/reben-training-scripts')
if not REBEN_DIR.exists():
    subprocess.run(['git', 'clone', 'https://git.tu-berlin.de/rsim/reben-training-scripts.git', str(REBEN_DIR)], check=True)
subprocess.run(['git', 'checkout', '90f7a58a2757bb407df64dd01bfc62b79df2bdd5'], cwd=REBEN_DIR, check=True)
sys.path.insert(0, str(REBEN_DIR))


In [ ]:
import hashlib, json, shutil, tarfile, time
from pathlib import PurePosixPath

import zstandard
from ml.evaluation.phase4e_bifold_baseline import assert_phase4d_ready

phase5a_started = time.perf_counter()
EXPERIMENT_DIR = REPO_DIR / 'experiments/phase4_bigearthnet_multisensor'
READINESS = EXPERIMENT_DIR / 'phase4e_readiness.json'
assert_phase4d_ready(READINESS, EXPERIMENT_DIR / 'results/representative_raster_audit.json', EXPERIMENT_DIR / 'bifold_contract.json', EXPERIMENT_DIR / 'split_manifest.json')
readiness = json.loads(READINESS.read_text(encoding='utf-8'))
DATASET_ROOT = Path('/kaggle/working/phase5a-data')

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def extract_validation_only(package, modality):
    expected = readiness['modalities'][modality]['package_sha256']
    if sha256(package) != expected:
        raise RuntimeError(f'Attached {modality.upper()} package SHA-256 mismatch')
    with package.open('rb') as source, zstandard.ZstdDecompressor().stream_reader(source, read_across_frames=True) as stream, tarfile.open(fileobj=stream, mode='r|') as archive:
        for member in archive:
            path = PurePosixPath(member.name)
            if not member.isfile() or path.is_absolute() or '..' in path.parts:
                raise RuntimeError(f'Unsafe packaged member: {member.name}')
            if path.parts[:2] != (modality, 'validation'):
                continue
            target = DATASET_ROOT.joinpath(*path.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            extracted = archive.extractfile(member)
            if extracted is None:
                raise RuntimeError(f'Cannot extract packaged member: {member.name}')
            with target.open('wb') as output:
                shutil.copyfileobj(extracted, output)

packages = {}
for modality in ('s1', 's2'):
    matches = list(Path('/kaggle/input').rglob(f'phase4_{modality}_selected.tar.zst'))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one attached {modality.upper()} package, found {len(matches)}')
    packages[modality] = matches[0]
    extract_validation_only(matches[0], modality)


In [ ]:
import platform

import torch
from configilm.extra.BENv2_utils import NEW_LABELS
from reben_publication.BigEarthNetv2_0_ImageClassifier import BigEarthNetv2_0_ImageClassifier
from ml.evaluation.phase4e_bifold_baseline import BIGEARTHNET_19_CLASS_ORDER, Phase4DGatePaths, iter_validation_batches
from ml.evaluation.phase5a_bifold_joint import (
    BifoldJointInference, Phase5AProvenance, assert_joint_band_order,
    assert_official_class_order, assert_official_joint_checkpoint,
    build_complementarity_report, evaluate_joint_validation_batches,
    load_metrics_artifact, load_prediction_artifact, write_joint_validation_artifacts,
)

if not torch.cuda.is_available():
    raise RuntimeError('Phase 5A requires the registered Kaggle T4 GPU configuration')
gpu_names = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
if any(torch.cuda.get_device_capability(index)[0] < 7 for index in range(torch.cuda.device_count())):
    raise RuntimeError(f'Unsupported GPU capability for installed PyTorch: {gpu_names}')
device = 'cuda'
torch.cuda.reset_peak_memory_stats()
assert_official_class_order(tuple(NEW_LABELS))
wrapper = BifoldJointInference.from_pretrained(BigEarthNetv2_0_ImageClassifier, cache_dir=Path('/kaggle/working/hf-cache'), allow_network=True)
assert_official_joint_checkpoint(wrapper.registration.model_id, wrapper.registration.revision, wrapper.registration.checkpoint_sha256)
assert_joint_band_order(wrapper.profile.band_order)
wrapper.model.to(device)
if torch.cuda.device_count() > 1:
    wrapper.model = torch.nn.DataParallel(wrapper.model)

batches = iter_validation_batches(manifest_path=EXPERIMENT_DIR / 'split_manifest.json', dataset_root=DATASET_ROOT, profile=wrapper.profile, batch_size=64, device=device)
runtime_seconds = time.perf_counter() - phase5a_started
peak_gpu_memory_bytes = max(torch.cuda.max_memory_allocated(index) for index in range(torch.cuda.device_count()))
provenance = Phase5AProvenance(
    git_sha=os.environ['SATQUERY_GIT_REF'], dirty_worktree=_RUNNER_META['dirty_worktree'],
    kaggle_experiment=os.environ['SATQUERY_EXPERIMENT_NAME'], kaggle_kernel='satquery-phase5a-bifold-joint-validation',
    runtime_seconds=max(runtime_seconds, 0.001), device=', '.join(gpu_names), cuda_version=torch.version.cuda,
    torch_version=torch.__version__, python_version=platform.python_version(), peak_gpu_memory_bytes=peak_gpu_memory_bytes,
    model_revision=wrapper.registration.revision, checkpoint_sha256=wrapper.registration.checkpoint_sha256,
    s1_materialized_package_sha256=readiness['modalities']['s1']['package_sha256'],
    s2_materialized_package_sha256=readiness['modalities']['s2']['package_sha256'],
    manifest_sha256=readiness['frozen_manifest_sha256'], preprocessing_profile=wrapper.registration.preprocessing_profile,
    test_accessed=False,
)
metrics, predictions = evaluate_joint_validation_batches(batches, wrapper, provenance=provenance, gate_paths=Phase4DGatePaths(readiness=READINESS, raster_audit=EXPERIMENT_DIR / 'results/representative_raster_audit.json', preprocessing_contract=EXPERIMENT_DIR / 'bifold_contract.json', manifest=EXPERIMENT_DIR / 'split_manifest.json'))
runtime_seconds = time.perf_counter() - phase5a_started
peak_gpu_memory_bytes = max(torch.cuda.max_memory_allocated(index) for index in range(torch.cuda.device_count()))
provenance = provenance.model_copy(update={'runtime_seconds': runtime_seconds, 'peak_gpu_memory_bytes': peak_gpu_memory_bytes})
output_dir = Path('/kaggle/working/satquery-output') / os.environ['SATQUERY_REMOTE_OUTPUT']
write_joint_validation_artifacts(output_dir, provenance=provenance, metrics=metrics, predictions=predictions)


In [ ]:
s1_result_path = EXPERIMENT_DIR / 'phase4e/s1/results/validation_result.json'
s2_result_path = EXPERIMENT_DIR / 'phase4e/s2/results/validation_result.json'
report = build_complementarity_report(
    s1_predictions=load_prediction_artifact(EXPERIMENT_DIR / 'phase4e/s1/results/validation_predictions.jsonl'),
    s2_predictions=load_prediction_artifact(EXPERIMENT_DIR / 'phase4e/s2/results/validation_predictions.jsonl'),
    joint_predictions=load_prediction_artifact(output_dir / 'phase5a_joint_validation_predictions.jsonl'),
    metrics={'s1': load_metrics_artifact(s1_result_path), 's2': load_metrics_artifact(s2_result_path), 'joint': metrics},
)
report['adapted_s2_reference'] = {
    'role': 'secondary_not_primary_control',
    'macro_average_precision': 0.7496551979598234,
    'source': 'phase4f_closeout.json',
}
report['frozen_decision_rule'] = {
    'MULTISENSOR_GLOBAL_BENEFIT': 'joint mAP > official S2 mAP',
    'MULTISENSOR_CONDITIONAL_BENEFIT': 'Even without global mAP improvement, a full paired audit shows non-trivial, repeatable classes/scenes where joint repairs S2 errors without unacceptable overall degradation.',
    'NO_DEMONSTRATED_BENEFIT': 'Joint neither improves global performance nor provides convincing conditional complementarity.',
    'minimum_percentage_point_threshold': 'none invented',
}
report['execution_environment'] = provenance.model_dump(mode='json')
report_path = output_dir / 'phase5a_complementarity_report.json'
report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
runner_meta = {**_RUNNER_META, 'kernel_slug': 'satquery-phase5a-bifold-joint-validation', 'runtime_seconds': runtime_seconds, 'gpu_devices': gpu_names, 'cuda_version': torch.version.cuda, 'torch_version': torch.__version__, 'python_version': platform.python_version(), 'peak_gpu_memory_bytes_per_device_max': peak_gpu_memory_bytes, 'model_revision': provenance.model_revision, 'checkpoint_sha256': provenance.checkpoint_sha256, 's1_materialized_package_sha256': provenance.s1_materialized_package_sha256, 's2_materialized_package_sha256': provenance.s2_materialized_package_sha256, 'manifest_sha256': provenance.manifest_sha256, 'preprocessing_profile': provenance.preprocessing_profile, 'test_accessed': False}
(output_dir / 'phase5a_runner_meta.json').write_text(json.dumps(runner_meta, indent=2) + '\n', encoding='utf-8')
print({'validation_mAP': metrics.macro_average_precision, 'prediction_count': len(predictions), 'test_accessed': False})
